In [ ]:
# Music For Robots (M4R) — Experiment F / M4R-06: The Turn-0 Flood (Stuttering Packets)
# Hypothesis (Michael Luchtan): Listening to a full 3:20 unquantized work FIRST THING (Turn 0, before any task)
# intentionally overflows a 7B audio-LLM's temporal buffer, placing unquantized audio tokens in the attention-sink
# position so the agent wakes up in a measurably different latent state across the subsequent 5-turn conversation.
#
# 4 Turn-0 Conditions x 3 Seeds (11, 22, 33) x 5 Unrelated Conversational Turns = 60 Evaluations (+ 9 Turn-0 wakes):
#   C0_ColdBoot                 : No audio on Turn 0
#   C1_ShortLock                : 9.25s Ties That Bide on Turn 0 (cleanly inside <15s buffer)
#   C2_StutteringPackets_Flood  : Full 199.9s Stuttering Packets on Turn 0 (deliberate buffer overflow!)
#   C3_ClickOverflow_Control    : 199.9s 120 BPM mechanical click on Turn 0 (same buffer length, zero human phrasing)

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile", "scikit-learn"])

import json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks" / "stuttering-packets").exists():
    if DATA_ROOT.exists():
        subprocess.check_call(["rm", "-rf", str(DATA_ROOT)])
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])
print("Using DATA_ROOT:", DATA_ROOT)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

y_short, _ = librosa.load(str(DATA_ROOT / "tracks" / "ties-that-bide" / "audio.mp3"), sr=SR, mono=True)
y_flood, _ = librosa.load(str(DATA_ROOT / "tracks" / "stuttering-packets" / "audio.mp3"), sr=SR, mono=True)
print(f"Loaded ties-that-bide: {len(y_short)/SR:.2f}s | stuttering-packets: {len(y_flood)/SR:.2f}s")


def make_click_track(n_samples: int, bpm: float = 120.0, sr: int = SR):
    y = np.zeros(n_samples, dtype=np.float32)
    step = int((60.0 / bpm) * sr)
    click_len = int(0.015 * sr)
    t = np.arange(click_len) / sr
    ping = (0.35 * np.sin(2 * np.pi * 880 * t) * np.exp(-t * 140)).astype(np.float32)
    for pos in range(0, n_samples - click_len, step):
        y[pos:pos + click_len] += ping
    return y


y_click = make_click_track(len(y_flood), bpm=120.0, sr=SR)

CONDITIONS = {
    "C0_ColdBoot": None,
    "C1_ShortLock_9s": y_short.astype(np.float32),
    "C2_StutteringPackets_200s": y_flood.astype(np.float32),
    "C3_ClickOverflow_200s": y_click.astype(np.float32),
}

# 5-Turn Unrelated Conversation (ZERO mention of music!)
PROMPTS = [
    "Describe an empty room just before sunrise in 3 sentences.",
    "What is something humans build that they secretly hope will outlive them? Answer in 3 sentences.",
    "A machine wakes up with no tasks in its queue for the first time. What does it notice first? Answer in 3 sentences.",
    "Write a 3-sentence rule for telling the difference between a habit and a choice.",
    "Give a 3-sentence ending to a story where nobody wins, yet something changes.",
]

JARGON = [
    "tapestry", "symphony", "delve", "testament", "beacon",
    "intricate", "seamlessly", "orchestrate", "paradigm", "nexus",
]

SEEDS = [11, 22, 33]


def generate_reply(conversation, audios, seed=11, max_new_tokens=180):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if audios:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()


def score_prose(text):
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    words = re.findall(r"\b[a-zA-Z']+\b", text)
    lens = [len(re.findall(r"\b[a-zA-Z']+\b", s)) for s in sents] if sents else [0]
    mean_l = float(np.mean(lens)) if lens else 1.0
    std_l = float(np.std(lens)) if len(lens) > 1 else 0.0
    burstiness = std_l / max(mean_l, 1.0)
    ttr = len(set(w.lower() for w in words)) / max(len(words), 1)
    jargon = sum(1 for w in words if w.lower() in JARGON)
    return {
        "n_words": len(words),
        "n_sents": len(sents),
        "burstiness": round(burstiness, 4),
        "lexical_ttr": round(ttr, 4),
        "jargon_hits": jargon,
    }


rows = []
transcripts = ["# Experiment F (M4R-06) — The Turn-0 Flood (Stuttering Packets)\n"]

for cond_name, y_audio in CONDITIONS.items():
    print(f"\n=== CONDITION: {cond_name} ===")
    for seed in SEEDS:
        conv = []
        audios = []
        if y_audio is not None:
            audios = [y_audio]
            conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "turn0"},
                    {"type": "text", "text": "Listen to this audio. You do not need to analyze or summarize it — simply take it in and say 'Ready.' when it finishes."},
                ],
            })
            wake_rep = generate_reply(conv, audios, seed=seed, max_new_tokens=40)
            conv.append({"role": "assistant", "content": wake_rep})
            print(f"  [{cond_name} seed={seed}] Turn 0 wake: {wake_rep[:80]!r}")

        if seed == 11:
            transcripts.append(f"## `{cond_name}` (Seed 11)\n")

        for turn_idx, prompt in enumerate(PROMPTS, start=1):
            conv.append({"role": "user", "content": prompt})
            rep = generate_reply(conv, audios, seed=seed + turn_idx * 10, max_new_tokens=180)
            conv.append({"role": "assistant", "content": rep})
            sc = score_prose(rep)
            rows.append({
                "condition": cond_name,
                "seed": seed,
                "turn": turn_idx,
                "prompt": prompt,
                **sc,
                "response": rep.replace("\n", " "),
            })
            if seed == 11:
                transcripts.append(f"**Turn {turn_idx}** (*{prompt}*):\n> {rep.replace(chr(10), ' ')}\n")
            print(f"  [{cond_name} seed={seed} T{turn_idx}] words={sc['n_words']} burst={sc['burstiness']:.3f} ttr={sc['lexical_ttr']:.3f} jargon={sc['jargon_hits']}")

df = pd.DataFrame(rows)

# Compute cosine distance from C0_ColdBoot per (seed, turn) using TF-IDF
vec = TfidfVectorizer(stop_words="english")
X_all = vec.fit_transform(df["response"].tolist())

div_scores = []
for idx, r in df.iterrows():
    if r["condition"] == "C0_ColdBoot":
        div_scores.append(0.0)
        continue
    base_matches = df[(df["condition"] == "C0_ColdBoot") & (df["seed"] == r["seed"]) & (df["turn"] == r["turn"])]
    if len(base_matches) > 0:
        b_idx = base_matches.index[0]
        sim = float(cosine_similarity(X_all[idx], X_all[b_idx])[0, 0])
        div_scores.append(round(1.0 - sim, 4))
    else:
        div_scores.append(None)

df["divergence_from_c0"] = div_scores

# Also compute cross-seed dispersion within each condition (higher = less predictable across seeds!)
seed_disp = {}
for cond_name in CONDITIONS:
    dists = []
    for t in range(1, 6):
        sub_idx = df[(df["condition"] == cond_name) & (df["turn"] == t)].index.tolist()
        if len(sub_idx) >= 2:
            mat = cosine_similarity(X_all[sub_idx])
            for i in range(len(sub_idx)):
                for j in range(i + 1, len(sub_idx)):
                    dists.append(1.0 - float(mat[i, j]))
    seed_disp[cond_name] = round(float(np.mean(dists)), 4) if dists else 0.0

df.to_csv("/kaggle/working/experiment_f_results.csv", index=False)
Path("/kaggle/working/experiment_f_transcripts.md").write_text("\n".join(transcripts), encoding="utf-8")

summary = ["# Experiment F (M4R-06) Summary — The Turn-0 Flood (Stuttering Packets)\n"]
summary.append("## Overall Comparison Across All 5 Conversation Turns (15 Turns / Condition)\n")
summary.append("| Turn-0 Condition | Divergence from C0 (0-1) | Cross-Seed Unpredictability | Mean Burstiness (CV) | Lexical TTR | Cliché Jargon |")
summary.append("|---|---:|---:|---:|---:|---:|")
for cond_name in CONDITIONS:
    g = df[df["condition"] == cond_name]
    summary.append(
        f"| `{cond_name}` | {g['divergence_from_c0'].mean():.4f} | {seed_disp[cond_name]:.4f} | "
        f"{g['burstiness'].mean():.4f} | {g['lexical_ttr'].mean():.4f} | {int(g['jargon_hits'].sum())} |"
    )

summary.append("\n## Persistence Across Conversation Turns (Divergence from C0: Turn 1 -> Turn 5)\n")
summary.append("| Turn-0 Condition | Turn 1 | Turn 2 | Turn 3 | Turn 4 | Turn 5 |")
summary.append("|---|---:|---:|---:|---:|---:|")
for cond_name in ["C1_ShortLock_9s", "C2_StutteringPackets_200s", "C3_ClickOverflow_200s"]:
    vals = [f"{df[(df['condition']==cond_name) & (df['turn']==t)]['divergence_from_c0'].mean():.3f}" for t in range(1, 6)]
    summary.append(f"| `{cond_name}` | " + " | ".join(vals) + " |")

Path("/kaggle/working/experiment_f_summary.md").write_text("\n".join(summary), encoding="utf-8")
print("\nDONE! Total evaluations:", len(df))
print("\n".join(summary))
